# Heart Disease – Enterprise NearestCentroid
## Step 1 – Imports & Seed

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
import joblib, time, os
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import NearestCentroid
np.random.seed(42)
print('Libraries loaded')

Libraries loaded


## Step 2 – Data Loading

In [2]:
df = pd.read_csv('data/heart_disease/heart.csv')
print(df.shape)
df.head()

(920, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## Step 3 – Preprocessing Pipeline

In [3]:
X = df.drop('target', axis=1)
y = df['target']
num_cols = [c for c in X.columns if c != 'thal']
cat_cols = ['thal']
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer()), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (736, 13), Test: (184, 13)


## Step 4 – Algorithm Pipeline Construction

In [4]:
pipeline = Pipeline([('pre', preprocessor), ('clf', NearestCentroid())])
print('Pipeline:', pipeline)

Pipeline: Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer()),
                                                                  ('sc',
                                                                   StandardScaler())]),
                                                  ['age', 'sex', 'cp',
                                                   'trestbps', 'chol', 'fbs',
                                                   'restecg', 'thalach',
                                                   'exang', 'oldpeak', 'slope',
                                                   'ca']),
                                                 ('cat',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='most_f

## Step 5 – Baseline Dummy Benchmark

In [5]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
dummy_acc = accuracy_score(y_test, dummy.predict(X_test))
print(f'Dummy accuracy: {dummy_acc:.4f}')

Dummy accuracy: 0.5543


## Step 6 – Training & Convergence Monitoring

In [6]:
pipeline.fit(X_train, y_train)
print('Training complete')

Training complete


## Step 7 – Test Set Evaluation

In [7]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred))

Test Accuracy: 0.8261
              precision    recall  f1-score   support

           0       0.79      0.83      0.81        82
           1       0.86      0.82      0.84       102

    accuracy                           0.83       184
   macro avg       0.82      0.83      0.82       184
weighted avg       0.83      0.83      0.83       184



## Step 8 – Visualization (Confusion Matrix)

In [8]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=80)
plt.show()
print('Confusion matrix saved.')

Confusion matrix saved.


## Step 9 – Cross-Validation Stability (3-Fold)

In [9]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
print(f'CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

CV Accuracy: 0.8228 ± 0.0078


## Step 10 – Production Serialization

In [10]:
os.makedirs('production_models', exist_ok=True)
pipeline.fit(X_train, y_train)
out_path = f'production_models/nc_heart_pipeline.joblib'
joblib.dump(pipeline, out_path, compress=3)
size_kb = os.path.getsize(out_path) / 1024
print(f'Pipeline saved to {out_path} ({size_kb:.1f} KB)')

Pipeline saved to production_models/nc_heart_pipeline.joblib (2.6 KB)


## Step 11 – Latency Smoke Test

In [11]:
X_sample = X_test.iloc[:100]
times = []
for _ in range(100):
    t0 = time.perf_counter()
    pipeline.predict(X_sample)
    times.append((time.perf_counter() - t0) * 1000)
avg_lat = np.mean(times)
print(f'Avg latency: {avg_lat:.2f} ms')
assert avg_lat < 50.0, f'Latency {avg_lat:.2f}ms exceeds 50ms SLA'
print('✅ Latency SLA passed')

Avg latency: 5.97 ms
✅ Latency SLA passed


## Step 12 – Advanced Analysis / Hyperparameter Tuning

In [12]:
from sklearn.neighbors import NearestCentroid
results = []
for metric in ['euclidean', 'manhattan']:
    try:
        pip_tmp = Pipeline([('pre', preprocessor),
                            ('clf', NearestCentroid(metric=metric))])
        sc = cross_val_score(pip_tmp, X, y, cv=3, scoring='accuracy').mean()
        results.append({'metric': metric, 'cv_accuracy': sc})
        print(f'  metric={metric}: {sc:.4f}')
    except Exception as e:
        print(f'  metric={metric} failed: {e}')
results_df = pd.DataFrame(results)
print(results_df)

  metric=euclidean: 0.7956
  metric=manhattan: 0.7446
      metric  cv_accuracy
0  euclidean     0.795626
1  manhattan     0.744576


## Executive Decision Matrix

| Metric | Dummy Baseline | NearestCentroid Model |
|--------|---------------|----------------|
| Accuracy | (dummy_acc) | (test_acc) |
| Strategy | most_frequent | NearestCentroid |
| CV Stability | N/A | cv_mean ± cv_std |
| Latency | N/A | avg_lat ms |
| Prod Ready | ❌ | ✅ |

**Recommendation**: Deploy NearestCentroid model — outperforms baseline with acceptable latency.